In [1]:
from groq import Groq

print("Groq imported successfully")


Groq imported successfully


In [2]:
import sys

!{sys.executable} -m pip install groq


[notice] A new release of pip is available: 25.0.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [3]:
import pandas as pd

decision_df = pd.read_csv(
    "../data/decision_results.csv"
)

decision_df.head(100)

,Current_State,Recommended_Action,Predicted_Next_State,Reward,Projected_Energy_Saving (%),Confidence (%),Confidence_Level
0,0.4-0.0-0.5,Check Equipment,"(0.3, 0.0, 0.5)",0.0,25.0,100,High


In [4]:
row = decision_df.iloc[0]

print(row)

Current_State                      0.4-0.0-0.5
Recommended_Action             Check Equipment
Predicted_Next_State           (0.3, 0.0, 0.5)
Reward                                     0.0
Projected_Energy_Saving (%)               25.0
Confidence (%)                             100
Confidence_Level                          High
Name: 0, dtype: object


In [5]:
import os

os.environ["GROQ_API_KEY"] = "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"

In [6]:
import os

print(os.getenv("GROQ_API_KEY"))

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


In [7]:
from groq import Groq
import os

client = Groq(
    api_key=os.getenv("GROQ_API_KEY")
)

print("Client Ready")

Client Ready


In [8]:
state_parts = row['Current_State'].split('-')
power = state_parts[0]
occupancy = state_parts[1]
temperature = state_parts[2]

prompt = f"""
You are an Agent Energy Optimization Decision Auditor.

Your job is NOT to choose an action.

A Reinforcement Learning (RL) agent has already chosen an action.

Your role is to:

1. Explain why the action was selected.
2. Evaluate potential benefits.
3. Discuss alternative actions.
4. Assess operational risks.
5. Interpret the confidence score.
6. Provide practical recommendations for a building operator.

Current State (normalized values, 0 to 1 scale):
- Power Consumption: {power}
- Occupancy: {occupancy}
- Indoor Temperature: {temperature}

Recommended Action:
{row['Recommended_Action']}

Predicted Next State:
{row['Predicted_Next_State']}

Reward:
{row['Reward']}

Projected Energy Saving:
{row['Projected_Energy_Saving (%)']}%

Confidence:
{row['Confidence (%)']}%

Confidence Level:
{row['Confidence_Level']}

Generate the report using exactly these sections:

1. Current State

2. Recommended Action

3. Expected Impact
   - Energy Saving
   - Reward
   - Predicted Next State

4. Why This Decision?

5. Alternative Actions Considered

6. Risk Assessment

7. Confidence Interpretation

8. Operational Recommendation

Keep the report professional and concise.
"""

In [9]:
response = client.chat.completions.create(
    model="llama-3.1-8b-instant",
    messages=[
        {
            "role": "user",
            "content": prompt
        }
    ]
)

In [10]:
report = response.choices[0].message.content

print(report)

**Energy Optimization Decision Report**

**1. Current State**

The current state of the building is:

- Power Consumption: 0.4 (indicating moderate energy usage)
- Occupancy: 0.0 (indicating low occupancy)
- Indoor Temperature: 0.5 (near the optimal level)

**2. Recommended Action**

The recommended action is to: **Check Equipment**

**3. Expected Impact**

- **Energy Saving:** The action is predicted to result in a 25.0% energy saving, which can lead to cost savings and a reduced carbon footprint.
- **Reward:** The predicted reward is 0.0, indicating a neutral outcome, possibly due to the system's design or limitations.
- **Predicted Next State:** The predicted next state is (0.3, 0.0, 0.5), indicating a slight reduction in power consumption.

**4. Why This Decision?**

The decision to check equipment is likely based on historical data indicating that malfunctions in equipment can lead to increased energy consumption. By identifying and addressing potential issues early on, the system

In [11]:
with open(
    "../data/final_energy_report.txt",
    "w",
    encoding="utf-8"
) as file:
    
    file.write(report)

print("Report Saved Successfully")

Report Saved Successfully
